In [1]:
!pip install langchain 

  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/557.4 kB ? eta -:--:--
   ------------------ --------------------- 262.1/557.4 kB ? eta -:--:--
   ---------------------------------------- 557.4/557.4 kB 1.7 MB/s  0:00:00
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
   ---------------------------------------- 0.0/671.1 kB ? eta -:--:--
   ------------------------------- -------- 524.3/671.1 kB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 671.1/671.1 kB 2.0 MB/s  0:00:00
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
Using cached distro-1.9.0-py3-none-

In [2]:
!pip install langchain-openai python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.4 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.8 MB/s  0:00:00
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------- ----- 786.4/918.7 kB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 918.7/918.7 kB 2.5 MB/s  0:00:00
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------------- ----------------------- 2/5 [tiktoken]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ----------

In [3]:
import warnings
warnings.filterwarnings("ignore",category=UserWarning)

In [21]:
import os
from pathlib import Path
from dotenv import load_dotenv

script_dir = Path().resolve()
env_path = script_dir / ".env"

load_dotenv(dotenv_path=env_path)

print("OPENAPI_KEY:", bool(os.getenv("OPENAPI_KEY")))

os.environ["OPENAPI_KEY"] = os.getenv("OPENAPI_KEY")


OPENAPI_KEY: True


In [10]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.5",
                        api_key=os.getenv("OPENAPI_KEY")
)

In [12]:
response = model.invoke("Hello, how are you?")
print("===== Model Response =====")
print(response)
print(response.content)

===== Model Response =====
content='Hello! I’m doing well, thanks for asking. How can I help you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 12, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-DxnI8UDQPPRK1O5x98ReknXfjEEsA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f2b9d-f293-7890-b8fa-1d71342ddb09-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 12, 'output_tokens': 21, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Hello! I’m doing well, thanks for asking. How can I help you today?


Promt Template

In [15]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

simple_temple = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} to a complete beginner in 2-3 sentences."
)

formatted = simple_temple.format(topic="Quantum Computing")
print("===== Formatted Prompt =====")
print(formatted)
print()

===== Formatted Prompt =====
Explain Quantum Computing to a complete beginner in 2-3 sentences.



In [18]:
chat_temple = ChatPromptTemplate.from_messages([
    ("system", " You are a helpful coding tutor, keep answers short and clear."),
    ("human", "Explain {concept} with a simple Python example.")
])

# system - setting the system role


message = chat_temple.format_messages(concept = "list comprehensions")
print("===== Formatted Chat Prompt =====")
for msg in message:
    print(f"{msg.type}: {msg.content}")

print()

===== Formatted Chat Prompt =====
system:  You are a helpful coding tutor, keep answers short and clear.
human: Explain list comprehensions with a simple Python example.



Chains - connecting the pieces with LCEL

In [20]:
from langchain_core.output_parsers import StrOutputParser

chain = chat_temple | model | StrOutputParser()

#strOutputParser() - to convert the model's output into a string format

result = chain.invoke({"concept": "AI Agentic Agents"})
print("===== Chain Result =====")
print(result)


===== Chain Result =====
An **AI agentic agent** is an AI system that can:

1. Receive a **goal**
2. Decide what to do next
3. Use **tools**
4. Remember results
5. Repeat until the goal is done

A normal chatbot mostly answers once.  
An agentic agent can take multiple steps toward a goal.

Example idea:

> Goal: “Find the capital of France and calculate 6 * 7.”

The agent decides it needs two tools: a lookup tool and a calculator.

```python
import re

# --- Tools ---
def lookup_country_capital(country):
    capitals = {
        "France": "Paris",
        "Germany": "Berlin",
        "Italy": "Rome"
    }
    return capitals.get(country, "Unknown")

def calculator(expression):
    # Simple safe-ish calculator for demo purposes
    if not re.match(r"^[0-9+\-*/ ().]+$", expression):
        return "Invalid expression"
    return eval(expression)


# --- Agent ---
class SimpleAgent:
    def __init__(self):
        self.memory = {}

    def think(self, goal):
        """Decide the next ac

Memory

In [22]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

memory = InMemoryChatMessageHistory()

memory.add_message(HumanMessage(content="My name is Selani ad I'm learning Langchain."))
memory.add_message(AIMessage(content="That's great! Langchain is a powerful framework for building applications with language models"))
memory.add_message(HumanMessage(content="What tools should I learn first"))
memory.add_message(AIMessage(content="I recommend starting with the basics of Langchain, such as prompts, chains, and memory. Once you're comfortable with those concepts, you can explore more advanced features like agents and tools."))

print("==== Memory Contents ====")
for msg in memory.messages:
    print(f"{msg.__class__.__name__}: {msg.content}")

    

==== Memory Contents ====
HumanMessage: My name is Selani ad I'm learning Langchain.
AIMessage: That's great! Langchain is a powerful framework for building applications with language models
HumanMessage: What tools should I learn first
AIMessage: I recommend starting with the basics of Langchain, such as prompts, chains, and memory. Once you're comfortable with those concepts, you can explore more advanced features like agents and tools.


In [23]:
chat_with_memory = ChatPromptTemplate.from_messages([
    ("system", "You are a helpfull tutor, Use the conversation history to personalize your responses."),
    ("placeholder","{history}"),
    ("human", "{question}")
])

chain_with_memory = chat_with_memory | model | StrOutputParser()

result = chain_with_memory.invoke({
    "history": memory.messages,
    "question": "What was my name again?    "
})

print("===== Chain with Memory Result =====")
print(result)

===== Chain with Memory Result =====
Your name is Selani.
